In [0]:
%run "/Workspace/Users/dnp50022@gmail.com/Retail-Sales-Data-Pipeline/databricks/notebooks/silver/service principle"

In [0]:

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import (col, trim, to_date, row_number, current_timestamp, lit,concat_ws,coalesce)

# COMMAND ----------


storage_account = "salesstorageproject"
container_name = "sales"
table_name = "shipping"

bronze_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze data/{table_name}/*"


In [0]:

# Read Transaction CSV from Bronze
df = spark.read.format("csv").option("header", "true").load(bronze_path)  
display(df)

In [0]:
df.columns

In [0]:
# CAST DATATYPES
df = (
    df.withColumn("shipment_id", col("shipment_id").cast("string"))
      .withColumn("order_id", col("order_id").cast("string"))
      .withColumn("shipment_status", col("shipment_status").cast("string"))
      .withColumn("shipment_date", to_date(col("shipment_date")))
      .withColumn("delivery_date", to_date(col("delivery_date")))
      .withColumn("carrier", col("carrier").cast("string"))
      .withColumn("modified_date", to_timestamp(col("modified_date")))
)

In [0]:
# PK FILTER (shipment_id must be valid)
df = df.filter(
    col("shipment_id").isNotNull() &
    (trim(col("shipment_id")) != "") &
    (col("shipment_id") != "0")
)


In [0]:
# FK VALIDATION
df = df.filter(col("order_id").isNotNull())

In [0]:
# STANDARDIZE TEXT
df = df.withColumn("shipment_status", upper(trim(col("shipment_status")))) \
       .withColumn("carrier", initcap(trim(col("carrier"))))


In [0]:
# DEDUPLICATION (Keep Latest Record)
w = Window.partitionBy("shipment_id").orderBy(
    col("modified_date").desc()
)

df = (
    df.withColumn("row_num", row_number().over(w))
      .filter(col("row_num") == 1)
      .drop("row_num")
)

In [0]:
# DERIVED COLUMN 

# Delivery Time (in days)
df = df.withColumn(
    "delivery_days",
    datediff(col("delivery_date"), col("shipment_date"))
)

# Delivery Status Flag
df = df.withColumn(
    "is_delivered",
    when(col("shipment_status") == "DELIVERED", 1).otherwise(0)
)

In [0]:
# ADD INGESTION TIMESTAMP
df = df.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
#  WRITE TO UNITY CATALOG
silver_table = "sales.silver.shipping"

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)

In [0]:
%sql
select * from sales.silver.shipping